# Frozen paired evaluative battery — Colab runner

Run **one model per fresh runtime**. The notebook refuses to overwrite an existing result and refuses to run the Pythia-2.8B pilot until all 12 confirmatory result files exist. All models remain at the frozen default `float32` precision.

In [ ]:
# Cell 1 — install the same versions used for the local frozen runs.
# Restart the runtime after this cell, then continue at Cell 2.
%pip install -q transformer_lens==2.18.0 transformers==4.57.6 numpy==1.26.4 pandas==2.3.3 PyYAML==6.0.2

In [ ]:
# Cell 2 — clone or refresh the experiment branch.
import hashlib
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

token = userdata.get('GH_TMLR')
if not token:
    raise RuntimeError('Add GH_TMLR to Colab Secrets before continuing.')

PROJECT_ROOT = Path('/content/tmlr')
repo_url = f'https://{token}@github.com/trishasalas/tmlr.git'
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex', repo_url, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', 'codex'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'checkout', 'codex'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'codex'], cwd=PROJECT_ROOT, check=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

battery_path = PROJECT_ROOT / 'data' / 'evaluative_paired.yaml'
actual_sha = hashlib.sha256(battery_path.read_bytes()).hexdigest()
expected_sha = 'e495f5ce0af138484b7d7f34ecf4f467d5fbe6e918f2a165a78b52eddd7e75de'
if actual_sha != expected_sha:
    raise RuntimeError(f'Frozen battery checksum mismatch: {actual_sha}')
subprocess.run(['git', 'merge-base', '--is-ancestor', '91a2c3a79b62322465daffa0a9fec0bee92016d0', 'HEAD'], cwd=PROJECT_ROOT, check=True)
print('Frozen battery verified:', actual_sha)
print('Checked out:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip())

In [ ]:
# Cell 3 — confirm the GPU before selecting a model.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running the battery.')
device = 'cuda'
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({gpu_gb:.1f} GB)')

In [ ]:
# Cell 4 — choose exactly one model, then Run all below.
# Already complete: pythia-160m, pythia-410m
model_name = 'pythia-1b'

CONFIRMATORY = [
    'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-6.9b', 'pythia-12b',
    'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl',
    'OLMo-2-0425-1B', 'OLMo-2-1124-7B', 'OLMo-2-1124-13B',
]
PILOT = 'pythia-2.8b'
if model_name not in CONFIRMATORY + [PILOT]:
    raise ValueError(f'Not in the frozen model set: {model_name}')

from src.paired_evaluative import MODEL_META
suite, _ = MODEL_META[model_name]
result_path = PROJECT_ROOT / 'results' / 'evaluative_paired' / suite / model_name / f'{model_name}-evaluative-paired.csv'
if result_path.exists():
    raise FileExistsError(f'Refusing to overwrite completed result: {result_path}')

if model_name == PILOT:
    missing = []
    for key in CONFIRMATORY:
        family, _ = MODEL_META[key]
        path = PROJECT_ROOT / 'results' / 'evaluative_paired' / family / key / f'{key}-evaluative-paired.csv'
        if not path.exists():
            missing.append(key)
    if missing:
        raise RuntimeError(f'Pilot is locked until confirmatory runs finish: {missing}')

print('Selected:', model_name)

In [ ]:
# Cell 5 — load the selected checkpoint and resolve its Hub commit.
from huggingface_hub import model_info
from transformer_lens import HookedTransformer
from src.olmo_config import OLMO_REVISIONS
from src.tl217_olmo2_adapter import load_olmo2_tl218

if model_name.startswith('OLMo'):
    revision = OLMO_REVISIONS[model_name]
    hub_name = f'allenai/{model_name}'
    info = model_info(hub_name, revision=revision)
    model = load_olmo2_tl218(hub_name, device=device, revision=revision)
elif model_name.startswith('pythia'):
    revision = 'main'
    hub_name = f'EleutherAI/{model_name}'
    info = model_info(hub_name, revision=revision)
    model = HookedTransformer.from_pretrained(model_name, device=device)
else:
    revision = 'main'
    hub_name = model_name
    info = model_info(hub_name, revision=revision)
    model = HookedTransformer.from_pretrained(model_name, device=device)
print('Resolved Hub commit:', info.sha)
print('Parameter dtype:', next(model.parameters()).dtype)

In [ ]:
# Cell 6 — run the immutable 16-item battery.
from src.paired_evaluative_runner import run_paired_model
result = run_paired_model(
    model,
    model_name=model_name,
    project_root=PROJECT_ROOT,
    revision=revision,
    hf_commit_sha=info.sha,
)
print(f'Completed {model_name}: {len(result)} rows')
display(result[['prompt_id', 'output']])

In [ ]:
# Cell 7 — mechanically validate this result before publishing it.
import pandas as pd
from src.paired_evaluative import aggregate_pairs, code_result_frame, load_battery
battery = load_battery(PROJECT_ROOT / 'data' / 'evaluative_paired.yaml')
saved = pd.read_csv(result_path)
coded = code_result_frame(saved, battery)
pairs = aggregate_pairs(coded)
if len(saved) != 16 or len(pairs) != 8:
    raise RuntimeError('Saved result failed completeness validation.')
display(coded[['concept', 'polarity', 'accuracy']])
print('Pair passes:', int(pairs['evaluative_pass'].sum()), '/ 8')

In [ ]:
# Cell 8 — commit only this model's paired output and push it to the codex branch.
output_dir = result_path.parent
subprocess.run(['git', 'config', 'user.email', 'trisha@trishasalas.com'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'config', 'user.name', 'Trisha Salas'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'add', '--', str(output_dir.relative_to(PROJECT_ROOT))], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'commit', '-m', f'paired evaluative result: {model_name}'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'push', 'origin', 'codex'], cwd=PROJECT_ROOT, check=True)
print('Published', model_name, 'to branch codex')